In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
#读取IDS2017数据集
filePath = '../../data/CIC-IDS2017/'
files=os.listdir(filePath)

In [3]:
files

['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
 'Friday-WorkingHours-Morning.pcap_ISCX.csv',
 'Monday-WorkingHours.pcap_ISCX.csv',
 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
 'Tuesday-WorkingHours.pcap_ISCX.csv',
 'Wednesday-workingHours.pcap_ISCX.csv']

In [4]:
# 合并数据集
df=pd.DataFrame()
for file in files:
    tmp_df=pd.read_csv(filePath+file)
    df = pd.concat([df,tmp_df],axis=0).reset_index(drop=True)

In [5]:
df.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [6]:
df.shape

(2830743, 79)

In [7]:
# 两列重复， Fwd Header Length和 Fwd Header Length.1
df = df.drop(' Fwd Header Length.1',axis=1) # 删除重复列

In [8]:
# 有空值 Flow Bytes/s
# 有inf值 Flow Bytes/s 、Flow Packets/s

In [9]:
# 判断是否存在空值
df.isna().any()[df.isna().any()]

Flow Bytes/s    True
dtype: bool

In [10]:
# 空值填充为0
df.fillna(value=0,inplace=True)

In [11]:
# 判断是否存在空值
df.isna().any()[df.isna().any()]

Series([], dtype: bool)

In [12]:
# 有inf值 Flow Bytes/s 、Flow Packets/s
# 判断是否存在inf值
for column in list(df.columns)[:-1]:
    if np.isinf(df[column]).any():
        print (column)

Flow Bytes/s
 Flow Packets/s


In [13]:
# 取最大值 需要排除inf 所以是索引是1
max_value = sorted(set(df[' Flow Packets/s'].values.tolist()),reverse=True)[1]
df[' Flow Packets/s'][np.isinf(df[' Flow Packets/s'])] = max_value

C:\Users\87211\AppData\Local\Temp\ipykernel_43380\2692027512.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[' Flow Packets/s'][np.isinf(df[' Flow Packets/s'])] = max_value


In [14]:
# 取最大值 需要排除inf 所以是索引是1
max_value = sorted(set(df['Flow Bytes/s'].values.tolist()),reverse=True)[1]
df['Flow Bytes/s'][np.isinf(df['Flow Bytes/s'])] = max_value

C:\Users\87211\AppData\Local\Temp\ipykernel_43380\1491039683.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Flow Bytes/s'][np.isinf(df['Flow Bytes/s'])] = max_value


In [15]:
# 判断是否存在inf值
for column in list(df.columns)[:-1]:
    if np.isinf(df[column]).any():
        print (column)

In [16]:
df.shape

(2830743, 78)

In [17]:
# np.save('./ids2017_X.npy',df.iloc[:,:-1])

In [18]:
#归一化处理 可选
from sklearn.preprocessing import  StandardScaler

standard_scaler_df=StandardScaler().fit_transform(df.iloc[:,:-1])#这里用的是标准化？
np.save('./ids2017_standard_scaler.npy',standard_scaler_df)

In [19]:
# 处理Label
Label = df[' Label']

In [20]:
list(pd.value_counts(Label).keys())

['BENIGN',
 'DoS Hulk',
 'PortScan',
 'DDoS',
 'DoS GoldenEye',
 'FTP-Patator',
 'SSH-Patator',
 'DoS slowloris',
 'DoS Slowhttptest',
 'Bot',
 'Web Attack � Brute Force',
 'Web Attack � XSS',
 'Infiltration',
 'Web Attack � Sql Injection',
 'Heartbleed']

In [21]:
# 使用sklearn之LabelEncoder将Label标准化
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
le.fit(Label)

LabelEncoder()

In [22]:
# 进行转化
label_bin=le.transform(Label)

In [23]:
# 显示标签转换后的结果
list(pd.value_counts(label_bin).keys())

[0, 4, 10, 2, 3, 7, 11, 6, 5, 1, 12, 14, 9, 13, 8]

In [24]:
# 显示数字所对应的标签
print(le.inverse_transform([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]))

['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack � Brute Force' 'Web Attack � Sql Injection'
 'Web Attack � XSS']


In [25]:
# 也可以使用replace方法进行替换，需要手动替换每个类
# label=df[' Label']
# label=label.replace(['BENIGN'],0)

In [26]:
# 保存前需要Reshape
label_bin = label_bin.reshape(label_bin.shape[0],1)
label_bin.shape

(2830743, 1)

In [27]:
# 保存标签
np.save('./ids2017_Y.npy',label_bin)

In [28]:
df[" Label"].value_counts()

 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [29]:
list(pd.value_counts(Label).keys())[0]

'BENIGN'

In [30]:
df[' Label'][df[' Label']==list(pd.value_counts(Label).keys())[0]].count()

2273097

In [31]:
list(pd.value_counts(Label).keys())[0]

'BENIGN'

In [32]:
subdf = pd.DataFrame()
for label in list(pd.value_counts(Label).keys()):
    df_temp = df[df[' Label'] == label]
    if(df[' Label'][df[' Label']==label].count()>10293):
        df_temp = df_temp.sample(frac=0.01)
    elif(df[' Label'][df[' Label']==label].count()<1507):
        df_temp = df_temp.sample(frac=1)
    else:
        df_temp = df_temp.sample(frac=0.1)
    df_temp.reset_index(drop=True, inplace=True)
    # df_temp[" Label"].value_counts()
    subdf = pd.concat([df_temp,subdf])
    subdf.reset_index(drop=True, inplace=True)
subdf[" Label"].value_counts()

 Label
BENIGN                        22731
DoS Hulk                       2311
PortScan                       1589
DDoS                           1280
DoS GoldenEye                  1029
FTP-Patator                     794
Web Attack � XSS                652
SSH-Patator                     590
DoS slowloris                   580
DoS Slowhttptest                550
Bot                             197
Web Attack � Brute Force        151
Infiltration                     36
Web Attack � Sql Injection       21
Heartbleed                       11
Name: count, dtype: int64

In [33]:
subdf.shape

(32522, 78)

In [34]:
# 归一化处理 可选
from sklearn.preprocessing import  StandardScaler

standard_scaler_df=StandardScaler().fit_transform(subdf.iloc[:,:-1])
standard_scaler_df=StandardScaler().fit_transform(df.iloc[:,:-1])
np.save('./sub_X.npy',standard_scaler_df)
np.save('./ids2017_X.npy')

In [35]:
Label = subdf[' Label']
# 使用sklearn之LabelEncoder将Label标准化
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
le.fit(Label)

LabelEncoder()

In [36]:
# 进行转化
label_le=le.transform(Label)

In [37]:
# 显示数字所对应的标签
print(le.inverse_transform(list(range(len(list(pd.value_counts(label_le).keys()))))))

['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack � Brute Force' 'Web Attack � Sql Injection'
 'Web Attack � XSS']


In [38]:
label_le = label_le.reshape(label_le.shape[0],1)
label_le.shape

(32522, 1)

In [39]:
np.save('./sub_Y.npy',label_le)

In [40]:
# from sklearn.decomposition import PCA

# pca_coordinates=PCA(n_components=2).fit_transform()

In [41]:
# 将第一个标签以外的所有标签设为1
binary_label = np.ones_like(label_le)
binary_label[label_le == label_le[0]] = 0

print("Binary Labels:", binary_label)

Binary Labels: [[0]
 [0]
 [0]
 ...
 [1]
 [1]
 [1]]


In [43]:
np.save('./sub_Y_2.npy',binary_label)